# Calculate the metrics from the csv

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def calculate_open_set_metrics_from_csv(csv_path, thresholds=None):
    """
    Calculates DIR @ FAR levels from a precomputed distance CSV.
    Assumes queryId and gallery column names follow 'DogID_SampleID' format.
    """
    df = pd.read_csv(csv_path, sep=';')
    
    # 1. Setup IDs and Labels
    query_ids = df['queryId'].values
    gallery_ids = df.columns[1:].values
    # Distances (Rows = Queries, Cols = Gallery)
    dist_mat = df.iloc[:, 1:].values 
    
    get_dog_label = lambda x: str(x).split('_')[0]
    
    q_labels = np.array([get_dog_label(i) for i in query_ids])
    g_labels = np.array([get_dog_label(i) for i in gallery_ids])
    
    # 2. Identify Known vs Unknown Queries
    # A query is 'known' if its Dog ID exists in the gallery set
    known_mask = np.array([q in g_labels for q in q_labels])
    unknown_mask = ~known_mask
    
    if not any(unknown_mask):
        print("Warning: No 'Unknown' dogs found in query set. FAR will always be 0.")

    # 3. Precompute best matches (Minimum Distance)
    # For every query, find the smallest distance and the ID of that dog
    best_dist = np.min(dist_mat, axis=1)
    best_idx = np.argmin(dist_mat, axis=1)
    best_match_label = g_labels[best_idx]
    
    # Logic for DIR: Distance < Threshold AND Label matches
    correct_match = (best_match_label == q_labels)
    
    # 4. Sweep Thresholds
    if thresholds is None:
        # Since your features are normalized, distances range from 0 to 2
        thresholds = np.linspace(0, 2, 1000)
        
    dir_curve = []
    far_curve = []
    
    for t in thresholds:
        # DIR: Fraction of KNOWN queries correctly identified with dist < t
        if any(known_mask):
            # Success = (is known) AND (dist < t) AND (correct dog)
            dir_val = np.sum((best_dist[known_mask] < t) & correct_match[known_mask]) / np.sum(known_mask)
        else:
            dir_val = 0
            
        # FAR: Fraction of UNKNOWN queries incorrectly accepted with dist < t
        if any(unknown_mask):
            # False Alarm = (is unknown) AND (any gallery dog is closer than t)
            far_val = np.sum(best_dist[unknown_mask] < t) / np.sum(unknown_mask)
        else:
            far_val = 0
            
        dir_curve.append(dir_val)
        far_curve.append(far_val)
        
    dir_curve = np.array(dir_curve)
    far_curve = np.array(far_curve)

    # 5. Extract specific operating points
    results = {}
    print("\n" + "="*40)
    print("OPEN SET PERFORMANCE (DIR @ FAR)")
    print("-"*40)
    
    for target_far in [0.01, 0.05, 0.1]: # 0.1%, 1%, 10%
        # Find index where far is closest to target
        idx = np.argmin(np.abs(far_curve - target_far))
        print(f"DIR @ {target_far*100:>4}% FAR: {dir_curve[idx]:.2%}")
        results[f"DIR@{target_far}"] = dir_curve[idx]
        
    print("="*40 + "\n")
    
    return thresholds, dir_curve, far_curve